In [1]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import partial_dependence
from sklearn.metrics import precision_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import os

random.seed(1337)

In [2]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import partial_dependence
from sklearn.metrics import precision_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import os

random.seed(1337)

### Define Survival Analysis Functions

def discrete_survival_probabilities(hazards, max_t):
    """
    Calculate PMF and CDF from discrete-time hazards.

    T = time until event (e.g., adoption)
    t is the discrete time period (e.g., year)
    pi = hazard at time t conditional on survival up to t-1
    pmf is different from hazard because its unconditional, and is the probability for everyone at the start
    cdf is the cumulative pmf up to time t, and is the probability of failure by time t
    survival is the probability of surviving past time t, and is 1 - cdf

    For example:
    PMF(5) = 0.08 → "8% of all subjects fail at time 5"
    π(5) = 0.15 → "15% of survivors to time 5 fail during time 5"
    
    Parameters:
    -----------
    hazards : array of π(t) for t = 1, 2, ..., max_t
    max_t   : maximum time period
    
    Returns:
    --------
    pmf : Pr(T = t) for each t
    cdf : Pr(T ≤ t) for each t  
    survival : Pr(T > t) for each t
    """
    
    pmf = np.zeros(max_t)
    cdf = np.zeros(max_t)
    survival = np.zeros(max_t)
    
    # Initialize
    survival_to_t = 1.0  # S(0) = 1, everyone "survives" to start
    cumulative_prob = 0.0
    
    for t in range(max_t):
        # PMF: probability of failure exactly at t
        # Pr(T = t) = π(t) × Pr(T ≥ t) = π(t) × S(t-1)
        pmf[t] = hazards[t] * survival_to_t
        
        # Update cumulative probability (CDF)
        cumulative_prob = cumulative_prob + pmf[t]
        cdf[t] = cumulative_prob
        
        # Update survival: S(t) = S(t-1) × (1 - π(t))
        survival_to_t = survival_to_t * (1 - hazards[t])
        survival[t] = survival_to_t
    
    return pmf, cdf, survival


# Alternative: Direct calculation without loop
def survival_function(hazards, t):
    """S(t) = ∏_{j=1}^{t} (1 - π(j))"""
    return np.prod(1 - hazards[:t])

def cdf_direct(hazards, t):
    """F(t) = 1 - S(t)"""
    return 1 - survival_function(hazards, t)

In [3]:
### Import Data and Train Model

boushey_2016_full = pd.read_stata(r"data/boushey2016.dta")

# Covariates
covariates = ["policycongruent","gub_election","elect2", "hvd_4yr", "fedcrime",
                "leg_dem_per_2pty","dem_governor","insession","propneighpol",
                "citidist","squire_prof86","citi6008","crimespendpc","crimespendpcsq",
                "violentthousand","pctwhite","stateincpercap","logpop","counter","counter2","counter3"]
boushey_2016 = boushey_2016_full[["state", "year", "billname", "dvadopt"] + covariates].dropna()

# Define X and y
X = boushey_2016[covariates].copy()
y = boushey_2016['dvadopt'].copy()

# Split policies, not individual observations
unique_policies = boushey_2016['billname'].unique()
train_policies, test_policies = train_test_split(
    unique_policies, test_size=0.2, random_state=1337
)

# Create train and test masks based on policy membership
train_mask = boushey_2016['billname'].isin(train_policies)
test_mask = boushey_2016['billname'].isin(test_policies)

# Split X and y using the masks
X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Use best hyperparameters from the random-split experiment
rf_model = RandomForestClassifier(
    bootstrap = True,
    ccp_alpha = 0.0,
    class_weight = None,
    criterion = 'gini',
    max_depth = 10,
    min_samples_leaf = 3,
    n_estimators = 500,
    random_state = 1337
)

# Fit rf model
rf_model.fit(X_train_scaled, y_train)

,n_estimators,500
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [6]:
# Calculate CDF for each state-policy combination in test data with MICE imputation
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

results = []

for policy in test_policies[1:2]:
    # Get all observations for this policy
    policy_data = boushey_2016[boushey_2016['billname'] == policy]
    
    # Loop through each state
    for state in policy_data['state'].unique():
        # Get state-specific data, sorted by year
        state_policy_data = policy_data[policy_data['state'] == state].sort_values('year')
        
        if len(state_policy_data) == 0:
            continue
        
        # Initialize variables for extending predictions
        extended_data = state_policy_data.copy()
        max_year = extended_data['year'].max()
        pred_adoption_time = None
        
        # Keep extending until CDF > 0.5 or max iterations
        max_extensions = 20
        for extension in range(max_extensions):
            # Get covariates and scale
            X_state_policy = extended_data[covariates]
            X_state_policy_scaled = scaler.transform(X_state_policy)
            
            # Get hazard predictions
            hazards = rf_model.predict_proba(X_state_policy_scaled)[:, 1]
            
            # Calculate CDF for each time point
            max_t = len(hazards)
            pmf, cdf, survival = discrete_survival_probabilities(hazards, max_t)
            
            # Check if CDF exceeds threshold
            if cdf[-1] > 0.5:
                adoption_indices = np.where(cdf > 0.5)[0]
                pred_adoption_time = extended_data['year'].values[adoption_indices[0]]
                break
            
            # If not, extend by one year using MICE imputation
            last_row = extended_data.iloc[[-1]].copy()
            last_row['year'] = max_year + extension + 1
            
            # Use MICE to impute covariates for the new year
            # Create a dataset with the last few years plus the new row
            imputation_window = extended_data.tail(5)  # Use last 5 years for context
            imputation_data = pd.concat([imputation_window, last_row])
            
            # Impute using IterativeImputer (MICE)
            imputer = IterativeImputer(random_state=1337, max_iter=10)
            imputed_values = imputer.fit_transform(imputation_data[covariates])
            
            # Update last row with imputed values
            last_row[covariates] = imputed_values[-1]
            last_row['dvadopt'] = 0  # Not yet adopted
            
            # Add to extended data
            extended_data = pd.concat([extended_data, last_row], ignore_index=True)
        
        # Find actual adoption time (if any)
        adoption_years = state_policy_data[state_policy_data['dvadopt'] == 1]['year'].values
        actual_adoption_time = adoption_years[0] if len(adoption_years) > 0 else None
        
        results.append({
            'policy': policy,
            'state': state,
            'years': extended_data['year'].values,
            'hazards': hazards,
            'cdf': cdf,
            'actual_adoption_time': actual_adoption_time,
            'pred_adoption_time': pred_adoption_time,
            'num_timepoints': len(extended_data)
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)
print(f"Calculated CDF trajectories for {len(results_df)} state-policy combinations")
print(f"States with predicted adoption (CDF > 0.5): {results_df['pred_adoption_time'].notna().sum()}")
print(f"States without predicted adoption: {results_df['pred_adoption_time'].isna().sum()}")

results_df

Calculated CDF trajectories for 47 state-policy combinations
States with predicted adoption (CDF > 0.5): 46
States without predicted adoption: 1


,policy,state,years,hazards,cdf,actual_adoption_time,pred_adoption_time,num_timepoints
0,amber,Alabama,"[1999, 2000, 2001, 2002, 2003, 2004]","[0.012007433045573047, 0.0743871273861827, 0.0...","[0.012007433045573047, 0.08550136198021363, 0....",2003,2004.0,6
1,amber,Arizona,"[1999, 2000, 2001, 2002, 2003, 2004]","[0.021791423696798616, 0.05494468088551571, 0....","[0.021791423696798616, 0.07553878176125267, 0....",2002,2004.0,6
2,amber,Arkansas,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...","[0.013927161221002717, 0.043006011755300805, 0...","[0.013927161221002717, 0.05633422131711511, 0....",2001,2013.0,15
3,amber,California,"[1999, 2000, 2001, 2002]","[0.13140467109022327, 0.2759974242631373, 0.23...","[0.13140467109022327, 0.37113474459631424, 0.5...",2002,2001.0,4
4,amber,Colorado,"[1999, 2000, 2001, 2002, 2003, 2004]","[0.027059278873125543, 0.07784881772080272, 0....","[0.027059278873125543, 0.10280156372527796, 0....",2002,2004.0,6
5,amber,Connecticut,"[1999, 2000, 2001, 2002, 2003, 2004]","[0.04584995201456077, 0.042207414746508236, 0....","[0.04584995201456077, 0.08612215882028293, 0.1...",2002,2004.0,6
6,amber,Delaware,"[1999, 2000, 2001, 2002, 2003, 2004]","[0.026669723428518533, 0.05955100641432684, 0....","[0.026669723428518533, 0.08463252097188534, 0....",2003,2004.0,6
7,amber,Florida,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...","[0.021470993989729428, 0.08659411207699175, 0....","[0.021470993989729428, 0.10620584440677014, 0....",2000,2007.0,9
8,amber,Georgia,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006]","[0.013090011493936985, 0.07574469233676108, 0....","[0.013090011493936985, 0.08784320493740515, 0....",2002,2006.0,8
9,amber,Idaho,"[1999, 2000, 2001, 2002, 2003, 2004, 2005]","[0.03163023260483274, 0.035151159254552436, 0....","[0.03163023260483274, 0.06566955251583417, 0.1...",2003,2005.0,7


In [7]:
# Filter for cases where both actual and predicted times are available
valid_predictions = results_df.dropna(subset=['actual_adoption_time', 'pred_adoption_time'])

# Calculate the difference (error) for each state-policy
valid_predictions['time_difference'] = valid_predictions['pred_adoption_time'] - valid_predictions['actual_adoption_time']

# Calculate average difference (mean error)
# mean_difference = valid_predictions['time_difference'].mean()

# Calculate average absolute difference (mean absolute error)
mean_absolute_difference = valid_predictions['time_difference'].abs().mean()

print(f"Number of valid predictions: {len(valid_predictions)}")
# print(f"Mean difference: {mean_difference:.2f} years")
print(f"Mean absolute difference: {mean_absolute_difference:.2f} years")
print(f"\nDistribution of errors:")
print(valid_predictions['time_difference'].describe())

Number of valid predictions: 46
Mean absolute difference: 2.28 years

Distribution of errors:
count    46.000000
mean      2.239130
std       2.861117
min      -1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max      12.000000
Name: time_difference, dtype: float64


In [ ]:
# Maybe the best evaluation metric is to just calculate the average number of years the prediction is off from the actual adoption time

# "Its not frequent that its predicting the exact year, as shown by our precision and recall estimates. However, this analysis shows that we are often only off by 1-2 years, which is still pretty great"






# Is it imputing with all states or just the state in question?
# With one policy probably wanna impute with all data. But wondering if we should impute just for each unique state using the data in aggregate